# CERES-Sorghum Experiment Notebook

End-to-end demonstration of the DSSAT-Python **CERES-Sorghum** model (`SGCER048`).

Topics covered:
1. Defining an experiment in JSON
2. Running the season-long simulation
3. Inspecting end-of-season outputs
4. Plotting daily trajectories (LAI, biomass, growth stage)
5. Cultivar coefficient (CUL) file I/O
6. Sensitivity analysis — effect of grain fill duration (`p5`)
7. Programmatic experiment modification

## 1 — Setup and imports

In [ ]:
import tempfile, os, pathlib
from dssat.io.converters.cul import write_cul, read_cul

cul_records = [{
    'VAR#':   'IB0026',
    'VRNAME': 'Pioneer 8251',
    'ECO#':   'DFAULT',
    'P1':  450.0, 'P2O': 12.5, 'P2R': 180.0,
    'P3':  100.0, 'P4':  50.0, 'P5':  480.0,
    'G1':    6.5, 'G2':  27.0, 'PHINT': 38.9,
}]

with tempfile.NamedTemporaryFile(suffix='.CUL', delete=False, mode='w') as f:
    cul_path = f.name

write_cul(cul_records, cul_path, model_code='SG')
print('Written CUL file:')
print(pathlib.Path(cul_path).read_text())

read_back = read_cul(cul_path, model_code='SG')
print('\nRead back from CUL file:')
print(json.dumps(read_back, indent=2))
os.unlink(cul_path)

## 2 — Experiment JSON

All inputs are expressed as a single JSON document.
The top-level keys mirror the DSSAT X-file sections.

In [ ]:
experiment = {
    "experiment_id": "SGEXP0001",
    "title": "CERES-Sorghum demo — West Africa",
    "crop": "SG",
    "model": "SGCER048",

    # --- cultivar coefficients (IB0026 ≈ Pioneer 8251)
    "cultivar": {
        "varno": "IB0026",
        "vrname": "Pioneer 8251",
        "p1":    450.0,
        "p2o":   12.5,
        "p2r":   180.0,
        "p3":    100.0,
        "p4":     50.0,
        "p5":    480.0,
        "g1":      6.5,
        "g2":     27.0,
        "phint":  38.9,
        "panth": 550.0,
        "tbase":   7.0,
        "topt":   30.0,
        "ropt":   26.0,
        "rue":     3.5
    },

    # --- planting
    "planting": {
        "date": "2020-06-15",
        "plant_population": 15.0,
        "row_spacing": 75.0,
        "sowing_depth": 4.0
    },

    # --- simulation controls
    "simulation": {
        "start_date": "2020-06-15",
        "end_date":   "2020-12-31",
        "water_balance": true,
        "nitrogen_cycle": false
    },

    # --- soil profile (deep Alfisol, West Africa)
    "soil": {
        "id": "IB00000001",
        "name": "Example Alfisol",
        "country": "BF",
        "layers": [
            {"depth_cm": 15,  "bd": 1.30, "ll": 0.065, "dul": 0.185, "sat": 0.38, "oc": 0.92, "ph": 6.2},
            {"depth_cm": 30,  "bd": 1.38, "ll": 0.075, "dul": 0.195, "sat": 0.37, "oc": 0.55, "ph": 6.3},
            {"depth_cm": 45,  "bd": 1.45, "ll": 0.080, "dul": 0.200, "sat": 0.36, "oc": 0.35, "ph": 6.4},
            {"depth_cm": 60,  "bd": 1.50, "ll": 0.085, "dul": 0.205, "sat": 0.35, "oc": 0.20, "ph": 6.5},
            {"depth_cm": 90,  "bd": 1.55, "ll": 0.090, "dul": 0.210, "sat": 0.34, "oc": 0.12, "ph": 6.6},
            {"depth_cm": 120, "bd": 1.60, "ll": 0.095, "dul": 0.215, "sat": 0.33, "oc": 0.08, "ph": 6.7}
        ],
        "initial_sw_fraction": 0.6
    },

    # --- daily weather (West Africa rainy season)
    "weather": {
        "station_id": "OUAG",
        "latitude":   12.35,
        "longitude":  -1.53,
        "altitude":  300.0,
        "co2_ppm":   380.0,
        "daily": [
            {"date": "2020-06-15", "tmax": 35.0, "tmin": 23.0, "srad": 22.0, "rain": 5.0},
            {"date": "2020-06-16", "tmax": 36.0, "tmin": 24.0, "srad": 20.0, "rain": 0.0},
            {"date": "2020-06-17", "tmax": 34.0, "tmin": 22.0, "srad": 18.0, "rain": 12.0},
            {"date": "2020-06-18", "tmax": 33.0, "tmin": 22.0, "srad": 16.0, "rain": 20.0},
            {"date": "2020-06-19", "tmax": 32.0, "tmin": 21.0, "srad": 19.0, "rain": 8.0}
        ]
    }
}

print(json.dumps(experiment, indent=2)[:600], '...')

## 3 — Running the simulation

The `Simulation` engine accepts the experiment dict directly.
Weather is extended to cover the full season using a synthetic loop
for this notebook demonstration.

In [ ]:
import numpy as np
from dssat.core.constants import NL, RUNINIT, SEASINIT, RATE, INTEGR
from dssat.core.types import SoilType, WeatherType, ControlType, SwitchType
from dssat.core.date_utils import incdat

# Build soil
soil = SoilType()
layers = experiment['soil']['layers']
soil.nlayr = len(layers)
depth_cum = 0.0
for i, lyr in enumerate(layers):
    thick = lyr['depth_cm'] - depth_cum
    soil.dlayr[i] = thick
    soil.ds[i]    = lyr['depth_cm']
    soil.ll[i]    = lyr['ll']
    soil.dul[i]   = lyr['dul']
    soil.sat[i]   = lyr['sat']
    soil.bd[i]    = lyr['bd']
    soil.shf[i]   = 1.0
    soil.kg2ppm[i]= 10.0 / (lyr['bd'] * thick)
    depth_cum     = lyr['depth_cm']

# SW initialisation at 60 % of DUL
sw_init = 0.6
sw = np.array([soil.dul[i]*sw_init + soil.ll[i]*(1-sw_init) for i in range(soil.nlayr)])
sw_full = np.zeros(NL)
sw_full[:soil.nlayr] = sw

# Build cultivar
cv_dict = experiment['cultivar']
cultivar = SorghumCultivar(
    varno=cv_dict.get('varno','IB0026'),
    vrname=cv_dict.get('vrname','Pioneer 8251'),
    p1=cv_dict['p1'], p2o=cv_dict['p2o'], p2r=cv_dict['p2r'],
    p3=cv_dict['p3'], p4=cv_dict['p4'], p5=cv_dict['p5'],
    g1=cv_dict['g1'], g2=cv_dict['g2'], phint=cv_dict['phint'],
    panth=cv_dict['panth'], tbase=cv_dict['tbase'], topt=cv_dict['topt'],
    ropt=cv_dict.get('ropt', 26.0), rue=cv_dict.get('rue', 3.5),
)

pltg = experiment['planting']
yrsim = 2020167  # Julian date: 2020-06-15

model = CeresSorghum(
    cultivar=cultivar,
    pltpop=pltg['plant_population'],
    sdepth=pltg['sowing_depth'],
    yrsim=yrsim,
    yrplt=yrsim,
)

iswitch = SwitchType()
iswitch.iswwat = 'N'  # no water balance for this demo
iswitch.iswnit = 'N'

ctrl = ControlType()
ctrl.yrsim = yrsim

# RUNINIT
ctrl.dynamic = RUNINIT; ctrl.yrdoy = yrsim
w = WeatherType(); w.tmax=35; w.tmin=23; w.srad=22; w.dayl=13; w.co2=380; w.tavg=29
model.run(ctrl, iswitch, soil, w, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

# SEASINIT
ctrl.dynamic = SEASINIT
model.run(ctrl, iswitch, soil, w, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

# Daily loop
records = []
for day in range(1, 201):
    yrdoy = incdat(yrsim, day)
    ctrl.yrdoy = yrdoy

    # Synthetic tropical weather (rainy season → dry)
    doy = yrdoy % 1000
    rain = max(0.0, 15.0 * np.sin((doy - 160) * np.pi / 100)) if doy < 260 else 0.0
    tmax = 35.0 - 0.05 * max(0, doy - 200)
    tmin = 22.0 - 0.02 * max(0, doy - 200)
    srad = 18.0 + 4.0 * np.cos((doy - 180) * np.pi / 90)

    w = WeatherType()
    w.tmax = tmax; w.tmin = tmin; w.srad = max(8, srad)
    w.rain = rain; w.dayl = 13.0; w.co2 = 380.0; w.tavg = (tmax+tmin)/2

    ctrl.dynamic = RATE
    model.run(ctrl, iswitch, soil, w, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
    ctrl.dynamic = INTEGR
    model.run(ctrl, iswitch, soil, w, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

    g = model.growth
    p = model.pheno
    records.append({
        'day': day, 'yrdoy': yrdoy,
        'istage': p.istage, 'xstage': p.xstage,
        'lai': g.lai, 'biomas': g.biomas,
        'yield_kg_ha': g.yield_, 'swfac': g.swfac,
        'tmax': tmax, 'tmin': tmin, 'srad': max(8, srad),
    })

    if p.istage == 6 and p.mdate > 0:
        print(f'Maturity reached on day {day} (YRDOY {yrdoy})')
        break

df = pd.DataFrame(records)
print(f'Simulation ran {len(df)} days')
print(df[['day','istage','xstage','lai','biomas','yield_kg_ha']].tail(10).to_string(index=False))

## 4 — End-of-season summary

In [ ]:
summary = model.summary()
print('\n=== CERES-Sorghum End-of-Season Summary ===')
for k, v in summary.items():
    print(f'  {k:<25} {v}')

## 5 — Daily trajectory plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('CERES-Sorghum — Daily Trajectories', fontsize=13, fontweight='bold')

# Stage progression
ax = axes[0, 0]
ax.plot(df['day'], df['xstage'], color='purple', lw=2)
ax.set_ylabel('Growth stage (xstage)')
ax.set_xlabel('Days after sowing')
ax.set_title('Phenological development')
stage_labels = {1:'E-juv',2:'Pan init',3:'Flag',4:'Anthesis',5:'Grain fill',6:'Maturity'}
for stage, label in stage_labels.items():
    rows = df[df['istage'] == stage]
    if not rows.empty:
        ax.axvline(rows['day'].iloc[0], color='grey', ls='--', alpha=0.5)
        ax.text(rows['day'].iloc[0]+0.5, 1.0, label, fontsize=7, rotation=90, va='bottom')
ax.set_ylim(0, 7)
ax.grid(True, alpha=0.3)

# LAI
ax = axes[0, 1]
ax.plot(df['day'], df['lai'], color='green', lw=2)
ax.set_ylabel('Leaf area index (m² m⁻²)')
ax.set_xlabel('Days after sowing')
ax.set_title('Leaf area index')
ax.grid(True, alpha=0.3)

# Above-ground biomass
ax = axes[1, 0]
ax.plot(df['day'], df['biomas'], color='saddlebrown', lw=2)
ax.set_ylabel('Biomass (g m⁻²)')
ax.set_xlabel('Days after sowing')
ax.set_title('Above-ground dry matter')
ax.grid(True, alpha=0.3)

# Grain yield accumulation
ax = axes[1, 1]
ax.plot(df['day'], df['yield_kg_ha'], color='goldenrod', lw=2)
ax.set_ylabel('Grain yield (kg ha⁻¹)')
ax.set_xlabel('Days after sowing')
ax.set_title('Grain yield accumulation')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sorghum_trajectories.png', bbox_inches='tight')
plt.show()

## 6 — Cultivar file I/O (CUL converter)

Round-trip: Python dict → DSSAT `.CUL` file → Python dict

In [ ]:
import tempfile, os, pathlib
from dssat.io.converters.cul import write_cul, read_cul

cul_records = [{
    'VAR#':   'IB0026',
    'VRNAME': 'Pioneer 8251',
    'ECO#':   'DFAULT',
    'P1':  450.0, 'P2O': 12.5, 'P2R': 180.0,
    'P3':  100.0, 'P4':  50.0, 'P5':  480.0,
    'G1':    6.5, 'G2':  27.0, 'PHINT': 38.9,
}]

with tempfile.NamedTemporaryFile(suffix='.CUL', delete=False, mode='w') as f:
    cul_path = f.name

write_cul(cul_records, cul_path, model_code='SG')
print('Written CUL file:')
print(pathlib.Path(cul_path).read_text())

read_back = read_cul(cul_path, model_code='SG')
print('\nRead back from CUL file:')
print(json.dumps(read_back, indent=2))
os.unlink(cul_path)

## 7 — Sensitivity analysis: effect of grain fill duration (P5)

In [ ]:
p5_values = [350, 400, 450, 500, 550, 600]
results = []

for p5 in p5_values:
    cv = SorghumCultivar(
        p1=450, p2o=12.5, p2r=180, p3=100, p4=50, p5=p5,
        g1=6.5, g2=27.0, phint=38.9, panth=550,
        tbase=7.0, topt=30.0, ropt=26.0, rue=3.5,
    )
    m = CeresSorghum(cultivar=cv, pltpop=15.0, sdepth=4.0, yrsim=yrsim, yrplt=yrsim)

    ctrl_r = ControlType(); ctrl_r.yrsim = yrsim
    ctrl_r.dynamic = RUNINIT; ctrl_r.yrdoy = yrsim
    w0 = WeatherType(); w0.tmax=35; w0.tmin=23; w0.srad=22; w0.dayl=13; w0.co2=380; w0.tavg=29
    m.run(ctrl_r, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
    ctrl_r.dynamic = SEASINIT
    m.run(ctrl_r, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

    for day in range(1, 201):
        yrdoy = incdat(yrsim, day)
        ctrl_r.yrdoy = yrdoy
        doy = yrdoy % 1000
        tmax_ = 35.0 - 0.05 * max(0, doy - 200)
        tmin_ = 22.0 - 0.02 * max(0, doy - 200)
        srad_ = max(8, 18.0 + 4.0*np.cos((doy-180)*np.pi/90))
        w_ = WeatherType()
        w_.tmax=tmax_; w_.tmin=tmin_; w_.srad=srad_; w_.dayl=13; w_.co2=380; w_.tavg=(tmax_+tmin_)/2
        ctrl_r.dynamic = RATE
        m.run(ctrl_r, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        ctrl_r.dynamic = INTEGR
        m.run(ctrl_r, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        if m.pheno.istage == 6 and m.pheno.mdate > 0:
            break

    s = m.summary()
    results.append({'p5': p5, 'yield_kg_ha': s['yield_kg_ha'], 'das_anthesis': day})
    print(f'P5={p5:4.0f}  yield={s["yield_kg_ha"]:7.1f} kg/ha')

df_sens = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_sens['p5'], df_sens['yield_kg_ha'], 'o-', color='goldenrod', lw=2, ms=8)
ax.set_xlabel('P5 — grain fill duration (°C d)')
ax.set_ylabel('Grain yield (kg ha⁻¹)')
ax.set_title('Sorghum yield sensitivity to P5')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('sorghum_p5_sensitivity.png', bbox_inches='tight')
plt.show()

## 8 — Programmatic experiment modification

Demonstrate looping over plant populations.

In [ ]:
import copy

pltpop_values = [5, 10, 15, 20, 25, 30]
pop_results = []

for pltpop in pltpop_values:
    exp2 = copy.deepcopy(experiment)
    exp2['planting']['plant_population'] = pltpop

    cv2 = SorghumCultivar(**{k: cv_dict[k] for k in
          ['p1','p2o','p2r','p3','p4','p5','g1','g2','phint','panth','tbase','topt','ropt','rue']})
    m2 = CeresSorghum(cultivar=cv2, pltpop=float(pltpop), sdepth=4.0, yrsim=yrsim, yrplt=yrsim)

    ctrl2 = ControlType(); ctrl2.yrsim = yrsim
    ctrl2.dynamic = RUNINIT; ctrl2.yrdoy = yrsim
    m2.run(ctrl2, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
    ctrl2.dynamic = SEASINIT
    m2.run(ctrl2, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

    for day in range(1, 201):
        yrdoy2 = incdat(yrsim, day)
        ctrl2.yrdoy = yrdoy2
        doy = yrdoy2 % 1000
        tmax_ = 35.0 - 0.05 * max(0, doy - 200)
        tmin_ = 22.0 - 0.02 * max(0, doy - 200)
        srad_ = max(8, 18.0 + 4.0*np.cos((doy-180)*np.pi/90))
        w_ = WeatherType()
        w_.tmax=tmax_; w_.tmin=tmin_; w_.srad=srad_; w_.dayl=13; w_.co2=380; w_.tavg=(tmax_+tmin_)/2
        ctrl2.dynamic = RATE
        m2.run(ctrl2, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        ctrl2.dynamic = INTEGR
        m2.run(ctrl2, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        if m2.pheno.istage == 6 and m2.pheno.mdate > 0:
            break

    s2 = m2.summary()
    pop_results.append({'pltpop': pltpop, 'yield_kg_ha': s2['yield_kg_ha'], 'biomass_kg_ha': s2['biomass_kg_ha']})

df_pop = pd.DataFrame(pop_results)
print(df_pop.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_pop['pltpop'], df_pop['yield_kg_ha'], 's-', color='goldenrod', lw=2, ms=8, label='Grain yield')
ax2 = ax.twinx()
ax2.plot(df_pop['pltpop'], df_pop['biomass_kg_ha'], '^--', color='saddlebrown', lw=2, ms=8, label='Biomass')
ax.set_xlabel('Plant population (plants m⁻²)')
ax.set_ylabel('Grain yield (kg ha⁻¹)', color='goldenrod')
ax2.set_ylabel('Biomass (kg ha⁻¹)', color='saddlebrown')
ax.set_title('Sorghum response to plant population')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, loc='lower right')
plt.tight_layout()
plt.savefig('sorghum_pltpop.png', bbox_inches='tight')
plt.show()